In [0]:
import traceback

In [0]:
%sql
-- Volume for config/ folder
CREATE EXTERNAL VOLUME IF NOT EXISTS alwaha_banking_dev_001.bronze.landing_config_volume
LOCATION 'abfss://landing@alwahabankingdev001.dfs.core.windows.net/config';
-- raw_files_volume
CREATE EXTERNAL VOLUME IF NOT EXISTS alwaha_banking_dev_001.bronze.raw_files_volume
LOCATION 'abfss://bronze@alwahabankingdev001.dfs.core.windows.net/raw_files';

In [0]:
%run "./01_raw_read_files_utils"

In [0]:
%run "./02_raw_write_to_delta_utils"

In [0]:
dbutils.widgets.text("p_adf_run_id", "default_run_id")
dbutils.widgets.text("p_storage_account", "alwahabankingdev001")
dbutils.widgets.text("p_container_name", "bronze")

adf_run_id = dbutils.widgets.get("p_adf_run_id")
storage_account = dbutils.widgets.get("p_storage_account")
container_name = dbutils.widgets.get("p_container_name")

spark.sql("USE CATALOG alwaha_banking_dev_001")
spark.sql("USE SCHEMA bronze")

control_path = "/Volumes/alwaha_banking_dev_001/bronze/landing_config_volume/control_table.json"
df_control_table = spark.read.option("multiline", "true").json(control_path)
active_sources = df_control_table.filter("is_active = true").collect()

try:
    for raw in active_sources:
        source_folder = raw["source_folder"]
        source_file = raw["source_file"]
        file_format = raw["file_format"]
        target_schema = raw["target_schema"]
        target_table = raw["target_table"]
        cluster_keys = raw["cluster_keys"]

        df = read_raw_data(
            storage_account = storage_account,
            folder_name = source_folder,
            file_name = source_file,
            file_format = file_format,
            adf_run_id = adf_run_id
        )

        write_delta_table(
            df = df,
            schema_name = target_schema,
            folder_name= source_folder,
            table_name = target_table,
            write_mode = "append",
            cluster_keys = cluster_keys,
            storage_account= storage_account
        )
except Exception as e:
    print("\n" + "="*60)
    print("ASLI ERROR YEH HAI:")
    print(e)
    print("="*60 + "\n")
    traceback.print_exc()